# Case Study 3 — Collections Prioritization

## Business question
**Who should collections contact first?**

You are given `data/case3_collections.csv.gz` — one row per customer who **already
defaulted** (`TARGET == 1` in the original data), with:
- Application + bureau features
- Full-history behavioural severity: `POS_FULL_MAX_DPD_DEF`, `CC_FULL_MAX_DPD_DEF`,
  `INSTAL_FULL_PCT_LATE`, `INSTAL_FULL_MAX_DAYS_LATE`
- `EXPOSURE` = current loan amount + external debt
- `DPD_SEVERITY`, `LATE_PCT` — summary severity measures
- **Targets:**
  - `RECOVERY_PRIORITY_SCORE` (0-100, continuous) — a weighted blend:
    `0.40 * percentile(EXPOSURE) + 0.35 * percentile(DPD_SEVERITY) + 0.25 * percentile(LATE_PCT)`
  - `PRIORITY_TIER` (`Tier 1 - Immediate` ... `Tier 4 - Low`) — quartiles of the above score

## Important framing
`RECOVERY_PRIORITY_SCORE` is **defined directly** from `EXPOSURE`, `DPD_SEVERITY`, and
`LATE_PCT` — so a model that uses those three columns as inputs will trivially reproduce
the formula. The interesting question is:

> **Can application/bureau/demographic features (known at origination, before any of this
> delinquency history existed) predict who will eventually become a high collections
> priority — *without* access to the current exposure/severity numbers?**

This matters operationally: those features are available from day one, while
`EXPOSURE`/`DPD_SEVERITY`/`LATE_PCT` only become known once the account is already
deteriorating.

## Deliverables
1. A model predicting `RECOVERY_PRIORITY_SCORE` (or `PRIORITY_TIER`) using **only**
   origination-time features (Task 2).
2. A second model using **all** features including the severity/exposure columns, used
   as a sanity check / upper bound (Task 4).
3. A discussion of how this would integrate into a collections workflow.

## Task 1 — Load & Explore

1. Load `case3_collections.csv.gz`. Confirm `PRIORITY_TIER` is roughly balanced (it's
   built from quartiles — why would that be a deliberate design choice for this label?).
2. Look at the distributions of `EXPOSURE`, `DPD_SEVERITY`, `LATE_PCT`, and
   `RECOVERY_PRIORITY_SCORE`. Are `EXPOSURE` and `DPD_SEVERITY` correlated with each
   other? What would that imply about the weighting in the score formula?

**Deliverables:**
- `PRIORITY_TIER` value counts confirming the quartile balance, with 1-2 sentences on why this design is deliberate
- Summary statistics/distributions for `EXPOSURE`, `DPD_SEVERITY`, `LATE_PCT`, `RECOVERY_PRIORITY_SCORE`
- Correlation analysis between exposure and severity components, with interpretation of the score's weighting

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(f"{DATA_DIR}/case3_collections.csv.gz")
# TODO: explore distributions and correlations


## Task 2 — Origination-Only Model

1. Define a feature set using **only** columns that would have been known at origination:
   application + bureau features (everything *except* `POS_FULL_*`, `CC_FULL_*`,
   `INSTAL_FULL_*`, `EXPOSURE`, `DPD_SEVERITY`, `LATE_PCT`, `RECOVERY_PRIORITY_SCORE`,
   `PRIORITY_TIER`).
2. Train a regression model to predict `RECOVERY_PRIORITY_SCORE` from this feature set.
3. Evaluate with **R²** and, more importantly, **Spearman rank correlation** between
   predicted and actual score. Why might rank correlation matter more than R² for a
   *prioritization* task (where what matters is the ordering of accounts, not the exact
   score value)?

**Deliverables:**
- Explicit list of origination-only feature columns used (and why each exclusion was made)
- Trained regression model predicting `RECOVERY_PRIORITY_SCORE`
- R² and Spearman rank correlation on the validation set, with 1-2 sentences on why rank correlation matters more for this task

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

# TODO: origination-only feature set, train, evaluate (R2 and Spearman)


## Task 3 — How Good Is "Good Enough"?

1. Using your Task 2 model's predicted scores, derive a predicted `PRIORITY_TIER` (e.g.
   by quartile-binning the predictions) and compare against the true `PRIORITY_TIER`
   with a confusion matrix.
2. If collections capacity only allows contacting the top 25% of accounts by predicted
   priority, what fraction of the *true* `Tier 1 - Immediate` accounts would actually
   get contacted? Is this good enough to be useful in practice?

**Deliverables:**
- Predicted `PRIORITY_TIER` (quartile-binned from predictions) vs. true `PRIORITY_TIER` confusion matrix
- Top-25% capacity capture-rate calculation for `Tier 1 - Immediate` accounts
- 1-2 sentence verdict on whether this is 'good enough' for practical use

In [ ]:
# TODO: predicted tiers vs true tiers, top-25% capture analysis


## Task 4 — Full-Feature Model (Sanity Check)

1. Now train the same kind of model **including** `EXPOSURE`, `DPD_SEVERITY`, `LATE_PCT`
   (and the other `*_FULL_*` columns).
2. Confirm it near-perfectly reproduces `RECOVERY_PRIORITY_SCORE` (as expected, since
   the score is a deterministic function of these). What is the practical use of this
   "model" vs. just computing the formula directly?
3. Compare its feature importances to Task 2's — does this change your answer to
   Task 1's question about `EXPOSURE` vs. `DPD_SEVERITY` weighting?

**Deliverables:**
- Trained full-feature regression model with R² and Spearman rank correlation
- Feature importance comparison vs. the Task 2 model
- Written interpretation of what this model is/isn't useful for in practice

In [ ]:
# TODO: full-feature model + comparison


## Task 5 — Stretch: Operational Design

Write a short design note (~150-250 words):

1. In production, would you score accounts for collections priority **at the moment
   they default** (using only origination-time features, per Task 2) or **periodically
   as new delinquency data accumulates** (using full features, per Task 4)? Could you
   do both, and how would they be used differently?
2. How would you validate that this prioritization actually improves recovery rates
   once deployed (e.g. an A/B test design)?
3. Are there fairness/compliance considerations in using demographic-adjacent features
   (e.g. `NAME_EDUCATION_TYPE`, `OCCUPATION_TYPE`, `ORGANIZATION_TYPE`) to decide who
   gets contacted first for debt collection?

**Deliverables:**
- A written design note (~150-250 words) covering scoring cadence, an A/B test validation design, and fairness/compliance considerations

*(Write your design note here)*